In [2]:
"""
preprocess.py - cleaning + train/test split for TCGA-BRCA subtype classification.

Run with:   python preprocess.py
Needs:      pandas, scikit-learn, matplotlib  (pip install pandas scikit-learn matplotlib)

Put the two raw files from UCSC Xena in data/raw/ (do NOT edit them):
  - expression (IlluminaHiSeq, usually called HiSeqV2)  -> EXPR_FILE
  - phenotype  (usually BRCA_clinicalMatrix)            -> PHENO_FILE
Adjust the names/columns in the CONFIG section if yours differ.
"""

import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # lets us save charts without opening a window
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [3]:
# ---------------------------------------------------------------- CONFIG
RAW_DIR = r"C:\Users\LENOVO\Desktop\Comp Bio Project\data\raw"
OUT_DIR = r"C:\Users\LENOVO\Desktop\Comp Bio Project\data\processed"
# CHANGE THIS to the exact name of your expression file (check the folder):
EXPR_FILE = os.path.join(RAW_DIR, "gene_expression_RNAseq.xlsx")
PHENO_FILE = os.path.join(RAW_DIR, "Phenotypes.xlsx")  # phenotype file
LABEL_COL = "PAM50Call_RNAseq"   # subtype column (CHECK this name in your file)
TEST_SIZE = 0.2
SEED = 42                        # fixed seed so results are reproducible
# -----------------------------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
log = []  # we collect notes here for your handoff note / slides


def load_table(path):
    """Read .xlsx/.xls with read_excel, anything else as tab-separated text.
    Excel files are slow to read, so a CSV copy is cached in OUT_DIR the
    first time and reused on later runs."""
    if path.lower().endswith((".xlsx", ".xls")):
        cache = os.path.join(OUT_DIR, os.path.basename(path) + ".cache.csv")
        if os.path.exists(cache):
            print(f"Using cached copy: {cache}")
            return pd.read_csv(cache, index_col=0)
        print(f"Reading Excel file (slow for big files): {path}")
        df = pd.read_excel(path, index_col=0)
        df.to_csv(cache)
        return df
    return pd.read_csv(path, sep="\t", index_col=0)


def note(msg):
    print(msg)
    log.append(msg)


# ---- 1. LOAD ----------------------------------------------------------
# Xena expression files: rows = genes, columns = samples, tab-separated.
expr = load_table(EXPR_FILE)
note(f"Expression raw shape (genes x samples): {expr.shape}")

# Transpose so rows = samples, columns = genes (what ML models expect).
expr = expr.T
note(f"Expression after transpose (samples x genes): {expr.shape}")

# Gene names look like "SYMBOL|EntrezID". A "?" symbol means the gene has
# no known name, so we drop those columns (they are uninterpretable).
unnamed = [c for c in expr.columns if str(c).startswith("?")]
expr = expr.drop(columns=unnamed)
note(f"Genes with unknown symbol ('?') dropped: {len(unnamed)}")
print("First 3 expression sample IDs:", list(expr.index[:3]))

# First column of the phenotype file = sample ID, so use it as the index.
pheno = load_table(PHENO_FILE)
note(f"Phenotype raw shape: {pheno.shape}")
print("First 3 phenotype sample IDs:", list(pheno.index[:3]))

# If the subtype column name is wrong, show likely candidates and stop.
if LABEL_COL not in pheno.columns:
    cands = [c for c in pheno.columns if "pam50" in c.lower()
             or "subtype" in c.lower()]
    print(f"\nColumn '{LABEL_COL}' not found. Possible label columns:")
    for c in cands:
        print("  ", c)
    raise SystemExit("Set LABEL_COL at the top of the script and rerun.")
print("Phenotype columns (check the label column is here):")
print(list(pheno.columns))

# ---- 2. LABELS --------------------------------------------------------
labels = pheno[LABEL_COL]

# Treat blanks / text "NA" as missing, strip stray spaces.
labels = labels.astype("string").str.strip()
labels = labels.replace({"": pd.NA, "NA": pd.NA, "nan": pd.NA,
                         "null": pd.NA, "NULL": pd.NA})
note(f"Samples with missing subtype label: {labels.isna().sum()}")
labels = labels.dropna()

print("Label values found (look for typos / odd values):")
print(labels.value_counts())

# ---- 3. MATCH SAMPLES BETWEEN THE TWO FILES ----------------------------
# Remove duplicated sample IDs if any (keep first).
expr = expr[~expr.index.duplicated(keep="first")]
labels = labels[~labels.index.duplicated(keep="first")]

common = expr.index.intersection(labels.index)
note(f"Samples in expression: {len(expr)}, with labels: {len(labels)}, "
     f"in both: {len(common)}")

X = expr.loc[common]
y = labels.loc[common].astype(str)

# ---- 4. CLEAN EXPRESSION ----------------------------------------------
# Genes with any missing value are dropped (simple and safe).
before = X.shape[1]
X = X.dropna(axis=1, how="any")
note(f"Genes dropped for missing values: {before - X.shape[1]}")

# Everything must be numeric.
X = X.apply(pd.to_numeric, errors="coerce")
X = X.dropna(axis=1, how="any")

# Duplicated gene columns (same name twice).
X = X.loc[:, ~X.columns.duplicated()]
note(f"Final cleaned matrix before split: {X.shape}")

# ---- 5. CLASS DISTRIBUTION (slide material) ----------------------------
counts = y.value_counts()
note("Class counts (all data):\n" + counts.to_string())
ax = counts.plot(kind="bar", title="Subtype distribution (TCGA-BRCA)")
ax.set_ylabel("Number of samples")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "class_distribution.png"), dpi=200)
plt.close()

# ---- 6. SPLIT FIRST (before any scaling / PCA) -------------------------
# stratify=y keeps the same subtype proportions in train and test,
# which matters because some subtypes (e.g. Normal) are rare.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

# ---- 7. REMOVE USELESS GENES (decided from TRAIN only) -----------------
# Genes that never change carry no information. We decide using the
# training set only, then apply the same gene list to the test set.
keep = X_train.columns[X_train.var(axis=0) > 0]
note(f"Genes with zero variance removed: {X_train.shape[1] - len(keep)}")
X_train = X_train[keep]
X_test = X_test[keep]  # same genes, same order

note(f"Train: {X_train.shape}, Test: {X_test.shape}")
note("Train class counts:\n" + y_train.value_counts().to_string())
note("Test class counts:\n" + y_test.value_counts().to_string())

# ---- 8. SAVE ----------------------------------------------------------
X_train.to_csv(os.path.join(OUT_DIR, "X_train.csv"))
X_test.to_csv(os.path.join(OUT_DIR, "X_test.csv"))
y_train.to_csv(os.path.join(OUT_DIR, "y_train.csv"), header=["subtype"])
y_test.to_csv(os.path.join(OUT_DIR, "y_test.csv"), header=["subtype"])

# Handoff note for your friend (also useful for slides).
with open(os.path.join(OUT_DIR, "handoff_note.txt"), "w") as f:
    f.write(f"Random seed: {SEED}, test size: {TEST_SIZE}\n")
    f.write("Data is NOT scaled and NOT PCA'd yet.\n")
    f.write("Fit Z-score + PCA on X_train only, then apply to X_test.\n\n")
    f.write("\n\n".join(log))

print("\nDone. Files saved in", OUT_DIR)

Reading Excel file (slow for big files): C:\Users\LENOVO\Desktop\Comp Bio Project\data\raw\gene_expression_RNAseq.xlsx
Expression raw shape (genes x samples): (20530, 1218)
Expression after transpose (samples x genes): (1218, 20530)
Genes with unknown symbol ('?') dropped: 29
First 3 expression sample IDs: ['TCGA-AR-A5QQ-01', 'TCGA-D8-A1JA-01', 'TCGA-BH-A0BQ-01']
Reading Excel file (slow for big files): C:\Users\LENOVO\Desktop\Comp Bio Project\data\raw\Phenotypes.xlsx
Phenotype raw shape: (1247, 193)
First 3 phenotype sample IDs: ['TCGA-3C-AAAU-01', 'TCGA-3C-AALI-01', 'TCGA-3C-AALJ-01']
Phenotype columns (check the label column is here):
['AJCC_Stage_nature2012', 'Age_at_Initial_Pathologic_Diagnosis_nature2012', 'CN_Clusters_nature2012', 'Converted_Stage_nature2012', 'Days_to_Date_of_Last_Contact_nature2012', 'Days_to_date_of_Death_nature2012', 'ER_Status_nature2012', 'Gender_nature2012', 'HER2_Final_Status_nature2012', 'Integrated_Clusters_no_exp__nature2012', 'Integrated_Clusters_uns

In [4]:
import pandas as pd

d = r"C:\Users\LENOVO\Desktop\Comp Bio Project\data\processed"
X_train = pd.read_csv(d + r"\X_train.csv", index_col=0)
X_test  = pd.read_csv(d + r"\X_test.csv",  index_col=0)
y_train = pd.read_csv(d + r"\y_train.csv", index_col=0)
y_test  = pd.read_csv(d + r"\y_test.csv",  index_col=0)

print(X_train.shape, X_test.shape)
print("Same genes, same order:", list(X_train.columns) == list(X_test.columns))
print("Rows match labels:", list(X_train.index) == list(y_train.index),
      list(X_test.index) == list(y_test.index))
print("Missing values:", X_train.isna().sum().sum(), X_test.isna().sum().sum())
print(y_train["subtype"].value_counts())
print(y_test["subtype"].value_counts())

(764, 20196) (192, 20196)
Same genes, same order: True
Rows match labels: True True
Missing values: 0 0
subtype
LumA      347
LumB      155
Basal     113
Normal     95
Her2       54
Name: count, dtype: int64
subtype
LumA      87
LumB      39
Basal     29
Normal    24
Her2      13
Name: count, dtype: int64
